In [0]:
from pyspark.sql.types import StructType, StringType, DoubleType
from pyspark.sql.functions import from_json, col, current_timestamp

print("Imports completed successfully")

Imports completed successfully


In [0]:
payments_schema = (
    StructType()
    .add("event_id", StringType())
    .add("payment_id", StringType())
    .add("order_id", StringType())
    .add("amount", DoubleType())
    .add("payment_method", StringType())
    .add("status", StringType())
    .add("timestamp", StringType())
)

print("Payments schema configured successfully")

Payments schema configured successfully


In [0]:
eh_conn_str = dbutils.secrets.get(
    "kv-scope",
    "eventhub-kafka-connection-string"
)

print("Event Hubs secret retrieved successfully")

Event Hubs secret retrieved successfully


In [0]:
eh_namespace = "evhns-nexpulse.servicebus.windows.net:9093"

kafka_sasl_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" password="{eh_conn_str}";'
)

print("Event Hubs Kafka configuration prepared")

Event Hubs Kafka configuration prepared


In [0]:
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", eh_namespace)
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", kafka_sasl_config)
    .option("subscribe", "payments")
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

print("Kafka/Event Hubs payments stream created successfully")

Kafka/Event Hubs payments stream created successfully


In [0]:
parsed = (
    raw_stream.select(
        from_json(
            col("value").cast("string"),
            payments_schema
        ).alias("data"),
        col("timestamp").alias("kafka_ingest_time"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset")
    )
    .select(
        "data.*",
        "kafka_ingest_time",
        "kafka_partition",
        "kafka_offset"
    )
    .withColumn(
        "bronze_loaded_at",
        current_timestamp()
    )
)

print("Payments JSON parsing and Bronze metadata configured successfully")

Payments JSON parsing and Bronze metadata configured successfully


In [0]:
bronze_path = (
    "abfss://bronze@adlsnexpulse01.dfs.core.windows.net/payments/"
)

checkpoint_path = (
    "abfss://bronze@adlsnexpulse01.dfs.core.windows.net/"
    "_checkpoints/payments/"
)

print("Bronze path:", bronze_path)
print("Checkpoint path:", checkpoint_path)

Bronze path: abfss://bronze@adlsnexpulse01.dfs.core.windows.net/payments/
Checkpoint path: abfss://bronze@adlsnexpulse01.dfs.core.windows.net/_checkpoints/payments/


In [0]:
query = (
    parsed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="30 seconds")
    .start(bronze_path)
)

print("Payments streaming query started successfully")
print("Query ID:", query.id)
print("Status:", query.status)